In [1]:
import pandas as pd

users = pd.read_csv("../data/junyi/raw/Info_UserData.csv")
logs = pd.read_csv("../data/junyi/raw/Log_Problem.csv")


In [2]:
engagement = logs.groupby("uuid").size().reset_index(name="activity_count")

engagement["engagement_score"] = (
    engagement["activity_count"] / engagement["activity_count"].max()
)

In [3]:
df = users.merge(engagement, on="uuid", how="left")
df.head()

,uuid,gender,points,badges_cnt,first_login_date_TW,user_grade,user_city,has_teacher_cnt,is_self_coach,has_student_cnt,belongs_to_class_cnt,has_class_cnt,activity_count,engagement_score
0,Y2RcCdmUJAYPUAIDElo4nE9KrkLLFzUIRdexG+ipaZQ=,NaN,18300,1,2019-01-24,1,kh,0,False,0,0,0,38,0.003217
1,lw/Rchfvl9q1UDaQRmeE6QJDJeXAK7nt56RvUvqxD/8=,NaN,6468,0,2019-01-24,1,ntpc,1,False,0,1,0,36,0.003047
2,ncVYyCw3osV77X9M+4NbI7LvBR5UiB4ix6Ca+baQArA=,male,4703,0,2019-01-24,1,tp,0,False,0,0,0,7,0.000593
3,J7Tbo1x2WtRpPuXeX7lWT9tkzWlSJeubl8UWjNmHh+4=,NaN,15525,1,2019-01-24,2,ntpc,0,False,0,0,0,75,0.006349
4,qijKzROzz1LmCaCxHJ3mOBOtjW/q4kW80tnpPmXHVYQ=,NaN,7945,0,2019-01-24,2,km,1,False,0,1,0,4,0.000339


In [4]:
df["churn_risk"] = df["engagement_score"].apply(
    lambda x: "high" if x < 0.3 else "low"
)

In [5]:
logs["timestamp_TW"] = pd.to_datetime(logs["timestamp_TW"])

# latest activity per user
last_activity = logs.groupby("uuid")["timestamp_TW"].max().reset_index()

# calculate inactivity days
import datetime

today = logs["timestamp_TW"].max()

last_activity["inactive_days"] = (today - last_activity["timestamp_TW"]).dt.days

last_activity.head()

,uuid,timestamp_TW,inactive_days
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,2018-12-22 08:30:00+00:00,221
1,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,2019-05-22 17:00:00+00:00,70
2,++E4TrlDYvGtPBg1edhkLXLEEbnfiAgAamPQ33vpW8M=,2018-09-16 19:45:00+00:00,318
3,++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=,2018-12-02 20:15:00+00:00,241
4,++GobOSWqrsaxoRg1bMN+T6biIJcgBXwuOH/ddq3DiU=,2019-06-15 09:45:00+00:00,46


In [6]:
df = df.merge(last_activity[["uuid", "inactive_days"]], on="uuid", how="left")

In [7]:
def compute_churn(row):
    if row["engagement_score"] < 0.3 or row["inactive_days"] > 30:
        return "high"
    return "low"

df["churn_risk"] = df.apply(compute_churn, axis=1)

In [8]:
def segment_user(score):
    if score > 0.7:
        return "highly_active"
    elif score > 0.3:
        return "moderate"
    else:
        return "low_active"

df["user_segment"] = df["engagement_score"].apply(segment_user)

In [9]:
reviews = pd.read_csv("../data/reviews/raw/reviews.csv")

sample_reviews = " ".join(reviews["Review"].dropna().head(20).tolist())
print(sample_reviews[:500])

good and interesting This class is very helpful to me. Currently, I'm still learning this class which makes up a lot of basic music knowledge. like!Prof and TAs are helpful and the discussion among students are quite active. Very rewarding learning experience! Easy to follow and includes a lot basic and important techniques to use sketchup. Really nice teacher!I could got the point eazliy but the v Great course - I recommend it for all, especially IT and Business Managers! One of the most useful


In [12]:
df.head(10)

,uuid,gender,points,badges_cnt,first_login_date_TW,user_grade,user_city,has_teacher_cnt,is_self_coach,has_student_cnt,belongs_to_class_cnt,has_class_cnt,activity_count,engagement_score,churn_risk,inactive_days,user_segment
0,Y2RcCdmUJAYPUAIDElo4nE9KrkLLFzUIRdexG+ipaZQ=,NaN,18300,1,2019-01-24,1,kh,0,False,0,0,0,38,0.003217,high,150,low_active
1,lw/Rchfvl9q1UDaQRmeE6QJDJeXAK7nt56RvUvqxD/8=,NaN,6468,0,2019-01-24,1,ntpc,1,False,0,1,0,36,0.003047,high,188,low_active
2,ncVYyCw3osV77X9M+4NbI7LvBR5UiB4ix6Ca+baQArA=,male,4703,0,2019-01-24,1,tp,0,False,0,0,0,7,0.000593,high,171,low_active
3,J7Tbo1x2WtRpPuXeX7lWT9tkzWlSJeubl8UWjNmHh+4=,NaN,15525,1,2019-01-24,2,ntpc,0,False,0,0,0,75,0.006349,high,188,low_active
4,qijKzROzz1LmCaCxHJ3mOBOtjW/q4kW80tnpPmXHVYQ=,NaN,7945,0,2019-01-24,2,km,1,False,0,1,0,4,0.000339,high,187,low_active
5,6Dm50WJCc9EQjr4Ar8SPukhnsTeS+kwX+9FyUI1o57k=,NaN,101353,11,2019-01-24,2,kh,1,False,0,1,0,134,0.011343,high,12,low_active
6,ADL8ZENEcGUW3bCQg4t8gA0tJR9R6H5OYwr+TCPb5oY=,male,7503,0,2019-01-24,2,kh,1,False,0,1,0,23,0.001947,high,188,low_active
7,mmshyl4As28DA/nu7s/ItJBHasl2F0bhPIGPI5Tylbo=,NaN,11535,4,2019-01-24,2,ntpc,1,False,0,1,0,13,0.001100,high,188,low_active
8,5eCvpCJiXsYg8jNhQAz8JltX6G0LSWpFb86a2GVuinA=,NaN,12825,2,2019-01-24,3,ntpc,1,False,0,1,0,30,0.002540,high,188,low_active
9,Gy++3drmDAso2O8WE+C0yUv5mMfUo7QujA6YEYi5GXA=,NaN,11700,1,2019-01-24,3,ntpc,1,False,0,1,0,35,0.002963,high,188,low_active


In [13]:
df["engagement_score"].describe()
df["inactive_days"].describe()

count    72758.000000
mean       139.102119
std        104.326856
min          0.000000
25%         45.000000
50%        111.000000
75%        227.000000
max        364.000000
Name: inactive_days, dtype: float64

In [14]:
eng_threshold = df["engagement_score"].quantile(0.3)
inactive_threshold = df["inactive_days"].quantile(0.7)

print(eng_threshold, inactive_threshold)

0.0018623550325912131 210.0


In [15]:
def compute_churn(row):
    if row["engagement_score"] < eng_threshold or row["inactive_days"] > inactive_threshold:
        return "high"
    return "low"

df["churn_risk"] = df.apply(compute_churn, axis=1)

In [16]:
df["churn_risk"].value_counts()

churn_risk
low     39594
high    33164
Name: count, dtype: int64